# Hash Tables, Dictionaries, and Sets

## The Connection

Dictionaries and sets are **built on top of hash tables**. The hash table is what makes them fast — it's the engine running under the hood.

---

## Why Does This Matter? O(1) Lookups

When you look up a value in a dictionary or check if something exists in a set, Python does it in **O(1) time** — meaning it's instant, regardless of how much data is stored.

This is possible because of hash tables:

1. Your key (e.g. `"Barcelona"`) is passed through a **hash function** → turned into an integer
2. That integer is used as a **list index** → Python jumps straight to that location
3. No searching needed — Python knows exactly where to look

Compare this to a plain list, where finding a value requires checking items one by one — **O(n)** time.

---

## The Key Idea

> A hash function turns an arbitrary key into a list index, giving you list-like speed without needing a numeric index.

This means instead of writing `data[0]`, `data[1]`... you can write `data["name"]`, `data["city"]` — and it's just as fast.

---

## Dictionaries vs Sets

Both use hash tables internally and work the same way algorithmically. The differences are minor:

| Feature | Dictionary | Set |
|---|---|---|
| Stores | Key **and** value | Key only |
| Ordered? | Yes (since Python 3.7) | No guaranteed order |
| Growth pattern | Slightly different | Slightly different |
| Use case | Map keys → values | Track unique items |

> **Note:** This is why `x in my_set` and `my_dict["key"]` are both O(1) — they both rely on the same hash table lookup mechanism.

---

## Summary

| Concept | Role |
|---|---|
| **Hash function** | Converts your key into a list index |
| **Hash table** | The underlying structure storing data at those indexes |
| **Dictionary** | A hash table that stores key-value pairs |
| **Set** | A hash table that stores keys only (no values) |
| **O(1) performance** | The payoff — instant lookups, no matter the size |


 ## What Is a Hash Table?

A hash table stores data in a chunk of memory made up of **buckets** (like slots in an array). To decide *where* to store your data, it uses a **hash function** — a formula that converts your key into a number, which becomes the bucket index.

---

## Inserting Data

### Step 1: Hash the Key

The key gets converted into a big integer by the hash function. Then a **mask** trims it down to fit within the available buckets.

**Example:** You have 8 buckets, so the mask is `0b111` (binary for 7):

```
hash = 28975
index = 28975 & 0b111 = 7  → goes into bucket #7
```

As the table grows, the mask gets bigger to cover more buckets.

### Step 2: Check the Bucket

Once you have an index, you check what's in that bucket:

| Bucket state | What happens |
|---|---|
| Empty | Insert your key + value here ✅ |
| Same key already there | It's a duplicate — do nothing 🔄 |
| Different key | **Collision!** Find a new bucket ❌ |

### Step 3: Handle Collisions (Probing)

When two keys land on the same bucket, Python finds the next available spot using this formula:

```python
perturb = hash(key)
i = perturb & mask

# Keep trying new spots:
perturb >>= 5
i = (i * 5 + perturb + 1) & mask
```

The clever part: it uses **higher-order bits** of the hash to vary the search path, so two keys that collide once are unlikely to keep colliding.

---

## A Real Collision Example

Imagine a hash function that uses the **first letter** of a city name:

| City | Calculation | Bucket |
|---|---|---|
| Rome | `ord('R') = 82` → `82 & 7 = 2` | #2 |
| Barcelona | `ord('B') = 66` → `66 & 7 = 2` | #2 ← collision! |

Both cities hash to bucket #2. So Barcelona gets moved to the next available spot via probing.

---

## Looking Up Data

Lookups follow the exact same steps as insertion:

1. Hash the key → get an index
2. Check if the key at that index matches yours
3. **Match** → return the value ✅
4. **No match** → keep probing with the same formula
5. **Empty bucket** → the key doesn't exist ❌
---

## Python's Memory Optimization

Python doesn't store key/value pairs *inside* the hash table directly. Instead:

- Key/value pairs go into a **separate regular array**
- The hash table only stores the **index** pointing into that array
This saves **30–95% memory**. As a bonus, Python dictionaries remember **insertion order** — guaranteed since Python 3.7.

---

## Key Concepts Summary

| Concept | What it means |
|---|---|
| **Hash function** | Converts a key into a number |
| **Mask** | Clips the hash to fit within the table size |
| **Collision** | Two keys map to the same bucket |
| **Probing** | Strategy for finding the next open bucket |
| **Load factor** | How full the table is — affects performance |
| **Empty buckets** | Most of a hash table's space is unused by design |

## What Makes an Object Hashable?

Most Python objects already know how to hash themselves:

| Type | How it's hashed |
|---|---|
| `int`, `float` | Based on the number's value |
| `str`, `tuple` | Based on their contents |
| `list` | Not hashable — lists can change, so their hash would change too |
| Custom class | Based on memory address (by default) |



## The Problem with Default Hashing in Custom Classes

By default, custom objects are hashed by their **memory address** — not their contents. This causes a surprise:

```python
p1 = Point(1, 1)
p2 = Point(1, 1)

set([p1, p2])           # → two entries! (they're at different memory addresses)
Point(1,1) in set([p1, p2])  # → False  ← unexpected!
```

Even though `p1` and `p2` have the same x and y values, Python sees them as different objects.


## The Fix: Custom Hash Function

Override `__hash__` to hash based on **content**, not memory:

```python
class Point(object):
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __hash__(self):
        return hash((self.x, self.y))  # hash based on values

    def __eq__(self, other):
        return self.x == other.x and self.y == other.y
```

Now Python compares by value:

```python
p1 = Point(1, 1)
p2 = Point(1, 1)

set([p1, p2])                   # → one entry
Point(1, 1) in set([p1, p2])   # → True
```

> **Note:** Only `p1` gets added to the set. `p2` is recognized as a duplicate and silently dropped.

---

## What Is Entropy?

Entropy measures **how evenly spread out** your hash values are.

Think of it like this: imagine you have 10 buckets and 100 items to hash. A **good** hash function spreads items evenly — roughly 10 per bucket. A **bad** hash function dumps everything into one bucket.

| Hash function quality | What happens | Performance |
|---|---|---|
| **Good** (high entropy) | Items spread evenly across buckets | O(1) — fast  |
| **Bad** (low entropy) | Everything piles into the same bucket | O(n) — slow  |

---

## A Real Example: Bad vs Good Hash

```python
class BadHash(str):
    def __hash__(self):
        return 42  # EVERY key gets the same hash → all collide!

class GoodHash(str):
    def __hash__(self):
        return ord(self[1]) + 26 * ord(self[0]) - 2619  # spread out evenly
```

**Benchmark results** for 1,000,000 lookups:

| Method | Time |
|---|---|
| `bad_dict` (all same hash) | 10.16 seconds  |
| `list` (linear search) | 8.96 seconds |
| `good_dict` (spread hashes) | 0.19 seconds  |

> A bad hash function can make a dictionary **54× slower** than a good one — even slower than a plain list!



## The Mask Problem: Finite Tables

Here's something subtle. Python only uses **some bits** of your hash value, based on the table size.

For a dictionary with 4 items, the mask is `0b111` (last 3 bits only):

```
hash(5)   = 5   → 5   & 0b111 = 5  ✓
hash(501) = 501 → 501 & 0b111 = 5  ← collision! same bucket as 5
```

So even if two numbers are very different, they can collide if their **last few bits** match.

> **Rule of thumb:** The bigger your dictionary, the more bits are used, and the fewer accidental collisions there are.

---

## How to Find the Mask for Your Dictionary

Python keeps hash tables at most **2/3 full** for performance, and sizes them in fixed steps:
`8 → 32 → 128 → 512 → 2,048 → ...`

**Example:** Storing 1,039 items:
1. Minimum buckets needed: `1,039 × (2/3 + 1) ≈ 1,731`
2. Next table size up: **2,048**
3. Mask: `bin(2048 - 1)` = `0b11111111111` (11 bits)



## Key Takeaways

- Default hashing for custom classes uses **memory address** — override `__hash__` if you need value-based equality
- Always define `__eq__` alongside `__hash__` — they go together
- A good hash function **spreads values evenly** (high entropy) → fewer collisions → fast O(1) lookups
- A bad hash function **piles values up** (low entropy) → many collisions → slow O(n) lookups
- For most cases, just use Python's built-in `hash()` on a tuple of the relevant fields — it's good enough

# Hashing in Data Engineering — Practical Guide

## Why Hashing Matters in Data Engineering

In data engineering, hashing is used for very different reasons than hash tables in Python. Instead of speeding up lookups, you use hashing to:

- Detect duplicate or changed records
- Partition data evenly across systems
- Anonymise sensitive data
- Build efficient data pipelines
---

## 1. Row-Level Hashing — Detecting Changes

The most common use case. You hash each row to create a **fingerprint**. If the fingerprint changes, the row changed.

In [4]:
import pandas as pd
import hashlib

df = pd.DataFrame({
    "id":    [1, 2, 3],
    "name":  ["Alice", "Bob", "Charlie"],
    "email": ["alice@mail.com", "bob@mail.com", "charlie@mail.com"]
})

def hash_row(row):
    value = "|".join(str(v) for v in row)
    return hashlib.md5(value.encode()).hexdigest()

df["row_hash"] = df.apply(hash_row, axis=1)
print(df)

#Now if any value in a row changes, its hash changes — you instantly know what's new or updated.

   id     name             email                          row_hash
0   1    Alice    alice@mail.com  5ce97e925bb4041aa1289509547b53b2
1   2      Bob      bob@mail.com  9b0c5b795ed25f0c1f9b54c603c5bba2
2   3  Charlie  charlie@mail.com  dcb37d470cbf2af4fc7af8733ece6ca2


## 2. Measuring Hash Quality (Collision Rate)
Before using a hash function on real data, check how evenly it distributes values.

In [6]:
import hashlib
import pandas as pd

df = pd.read_csv("your_data.csv")

# Hash the 'id' column
df["hash_value"] = df["id"].astype(str).apply(
    lambda x: int(hashlib.md5(x.encode()).hexdigest(), 16)
)

# Simulate 10 buckets
num_buckets = 10
df["bucket"] = df["hash_value"] % num_buckets

# Measure distribution
bucket_counts = df["bucket"].value_counts().sort_index()
print(bucket_counts)

# Measure collision rate
total = len(df)
unique_hashes = df["hash_value"].nunique()
collision_rate = (total - unique_hashes) / total * 100
print(f"Collision rate: {collision_rate:.2f}%")

bucket
0    2
1    3
2    4
3    4
4    3
5    3
6    1
7    2
8    1
9    2
Name: count, dtype: int64
Collision rate: 0.00%


**What good looks like:**

| Metric | Good | Bad |
|---|---|---|
| Collision rate | < 1% | > 5% |
| Bucket distribution | Roughly equal counts | One bucket has 80% of rows |
| Unique hashes | ≈ total rows | Much fewer than total rows |

## 3. Partition Hashing — Spreading Data Evenly

In systems like Spark, Kafka, or databases, hashing decides which **partition** or **node** a record goes to.

In [7]:
num_partitions = 4

df["partition"] = df["id"].apply(
    lambda x: hash(str(x)) % num_partitions
)

# Check balance
print(df["partition"].value_counts())

partition
1    8
2    7
3    7
0    3
Name: count, dtype: int64


**Ideal output** — each partition gets roughly the same number of rows:

```
0    250
1    252
2    248
3    250
```

**Red flag** — a skewed partition (data skew):

```
0    850   ← overloaded
1     50
2     60
3     40
```

> Data skew is one of the most common performance killers in Spark and distributed systems. Hashing helps you detect it early.

## 4. Choosing the Right Hash Function

| Function | Speed | Collision safety | Use case |
|---|---|---|---|
| `hash()` (Python built-in) | ⚡ Fastest | Low | In-memory lookups only |
| `MD5` | Fast | Medium | Row fingerprinting, deduplication |
| `SHA-256` | Slower | High | PII anonymisation, data integrity |
| `MurmurHash` | ⚡ Very fast | Medium | Partitioning in Spark/Kafka |
| `xxHash` | ⚡ Fastest | Medium | High-volume pipelines |

> **Never use Python's `hash()` across runs or systems** — it changes between Python sessions (randomised by default). Use `hashlib` for anything stored or shared.

## 5. Measuring Entropy on Your Dataset

Entropy tells you how evenly your hash function distributes values — higher is better.

In [8]:
import pandas as pd
import numpy as np
import hashlib

df = pd.read_csv("your_data.csv")

df["hash_bucket"] = df["id"].apply(
    lambda x: int(hashlib.md5(str(x).encode()).hexdigest(), 16) % 256
)

# Calculate entropy
counts = df["hash_bucket"].value_counts(normalize=True)
entropy = -np.sum(counts * np.log2(counts))
max_entropy = np.log2(256)  # perfect entropy for 256 buckets

print(f"Entropy:      {entropy:.2f} bits")
print(f"Max possible: {max_entropy:.2f} bits")
print(f"Efficiency:   {entropy / max_entropy * 100:.1f}%")

Entropy:      4.48 bits
Max possible: 8.00 bits
Efficiency:   56.0%


**Interpreting results:**

| Efficiency | Meaning |
|---|---|
| 95–100% | Excellent — very even distribution |
| 80–94% | Acceptable |
| Below 80% | Poor — expect slowdowns and skew |

The problem is not hash function — it's a mismatch between data size and bucket count. There is only 25 rows spread across 256 buckets, so most buckets are empty, which kills entropy.

The Real Problem: Too Many Buckets for the data
25 rows  ÷  256 buckets  =  most buckets are EMPTY
Empty buckets drag entropy down — they contribute nothing but make the "spread" look terrible. It's like judging how well 25 people fill a stadium.

More data — entropy improves naturally as rows grow
A composite key — hash on multiple columns instead of just id

In [9]:
import pandas as pd
import numpy as np
import hashlib

df = pd.read_csv("your_data.csv")

# ── Rule of thumb: buckets should not exceed ~2x your row count ──
num_rows = len(df)
num_buckets = max(8, 2 ** int(np.log2(num_rows)))  # largest power of 2 ≤ row count

print(f"Rows: {num_rows}, Buckets chosen: {num_buckets}")

df["hash_bucket"] = df["id"].apply(
    lambda x: int(hashlib.md5(str(x).encode()).hexdigest(), 16) % num_buckets
)

# ── Entropy calculation ──
counts = df["hash_bucket"].value_counts(normalize=True)
entropy = -np.sum(counts * np.log2(counts))
max_entropy = np.log2(num_buckets)

print(f"\nEntropy:      {entropy:.2f} bits")
print(f"Max possible: {max_entropy:.2f} bits")
print(f"Efficiency:   {entropy / max_entropy * 100:.1f}%")

# ── Bucket distribution (spot skew visually) ──
print("\nBucket distribution:")
print(df["hash_bucket"].value_counts().sort_index().to_string())

# ── Collision check ──
total = len(df)
unique_hashes = df["hash_bucket"].nunique()
collision_rate = (total - unique_hashes) / total * 100
print(f"\nCollision rate: {collision_rate:.2f}%")
print(f"Filled buckets: {unique_hashes} / {num_buckets}")

Rows: 25, Buckets chosen: 16

Entropy:      3.33 bits
Max possible: 4.00 bits
Efficiency:   83.3%

Bucket distribution:
hash_bucket
0     2
3     4
4     2
5     1
6     2
9     3
10    1
11    2
12    4
13    2
15    2

Collision rate: 56.00%
Filled buckets: 11 / 16


## Quick Reference — When to Use What

| Situation | What to use |
|---|---|
| Detect changed rows | MD5 or SHA-256 row hash |
| Daily incremental loads | Hash-based CDC (compare old vs new hash) |
| Anonymise PII | SHA-256 with salt |
| Balance Spark partitions | MurmurHash or MD5 % num_partitions |
| Measure data skew | Bucket distribution + entropy score |
| Deduplicate records | Hash the key columns, find duplicates |

# Dictionaries vs Sets in Data Engineering

## What are they?

| | Dictionary | Set                               |
|---|---|-----------------------------------|
| **Stores** | Key → Value pairs | Unique values only                |
| **Lookup speed** | O(1) | O(1)                              |
| **Allows duplicates** |  keys must be unique | No                                |
| **Ordered** | Python 3.7+ | No                                |
| **Use for** | Mapping / enrichment | Membership checks / deduplication |

Both use **hashing** under the hood — that's why both are O(1) for lookups.



## Dictionary — O(1)
**"Map one thing to another"**

✅ Use when:
- You need to **enrich records** (join a lookup table)
- You need to **count or aggregate** values
- You need to **group** records by a key
- You need to **cache** results of expensive lookups


Avoid when:
- You only need to check **existence** (use a set — lighter)
- Keys are **not unique** (dict will silently overwrite duplicates)

---

## Set — O(1)
**"Track what you've seen"**

Use when:
- **Deduplication** — remove duplicate records
- **Membership checks** — has this ID been processed?
- **Comparing two datasets** — what's new, missing, or common?
- **Filtering** — exclude a known list of values


> **Rule of thumb:**
> - Need to **map** something → Dictionary
> - Need to **track or compare** something → Set
> - Need **both** in a pipeline → Use them together